
# V33 LIVE — mE5 + Twitter-XLM-R Super-Ensemble

**Protected live Kaggle champion: V14 = 0.586**

This notebook uses ONLY the Kaggle-accepted LIVE test IDs:
`BICH_0003, BICH_0006, BICH_0015, BICH_0016, ...`

## New models

1. `intfloat/multilingual-e5-base`
2. `cardiffnlp/twitter-xlm-roberta-base`

Both are trained with:

- 5-fold grouped OOF on LIVE train + validation
- hierarchical 7-label head
- competition-shaped loss
- hard-negative downweighting for attributes
- LLRD
- light FGM adversarial training
- snapshot averaging

## Reused LIVE signals

- V32 Stage-1
- V32 Stage-2 attention model
- V32 lexical expert
- V15 specialist
- V17 BanglaBERT / XLM-R toxic experts
- V18 MuRIL
- V19 mDeBERTa toxic
- V23 conditional attributes
- V25 BanglaBERT Large toxic
- V14 probability family

A regularized stacker is fit on **3297 train OOF rows only**.
The 500 LIVE validation rows are used for final evaluation/calibration.

Do not submit automatically. Send `v33_results.zip` to ChatGPT first.


In [ ]:

# 1. Upload V33 LIVE data
from google.colab import files
import pathlib, zipfile

uploaded=files.upload()
assert "v33_LIVE_data.zip" in uploaded

DATA_DIR=pathlib.Path("/content/v33_data")
DATA_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile("v33_LIVE_data.zip") as z:
    z.extractall(DATA_DIR)

print("Extracted:")
for p in sorted(DATA_DIR.iterdir()):
    print(" ",p.name,f"{p.stat().st_size:,} bytes")


In [ ]:

# 2. Install + checkpoint preflight
import sys,subprocess

subprocess.check_call([
    sys.executable,"-m","pip","install","-q",
    "transformers==4.57.6","tokenizers==0.22.1","sentencepiece==0.2.1",
    "scikit-learn","scipy","pandas","numpy","tqdm","safetensors"
])

from transformers import AutoConfig,AutoTokenizer

CHECKPOINTS={
    "me5":"intfloat/multilingual-e5-base",
    "twitter":"cardiffnlp/twitter-xlm-roberta-base",
}

for k,v in CHECKPOINTS.items():
    print("Checking",k,v)
    _=AutoConfig.from_pretrained(v)
    _=AutoTokenizer.from_pretrained(v)

print("✅ Both new checkpoints accessible")


In [ ]:

# 3. Imports, strict LIVE verification, preprocessing
import gc,re,math,json,random,unicodedata,zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader
from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,AutoModel,get_linear_schedule_with_warmup
)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

SEED=2026
N_FOLDS=5
EPOCHS=3
SNAPSHOT_EPOCHS=[2,3]
MAX_LENGTH=128
BATCH=8
ACCUM=2
EVAL_BATCH=32
BASE_LR=1.2e-5
HEAD_LR=3.0e-5
LAYER_DECAY=.90
WEIGHT_DECAY=.01
WARMUP_RATIO=.10
FGM_EPS=.20

LABELS=[
    "is_toxic","implicit_toxicity","sarcasm","contains_profanity",
    "threatening","identity_attack","sexual_content"
]
LI={l:i for i,l in enumerate(LABELS)}
ATTRS=LABELS[1:]

CHECKPOINTS={
    "me5":"intfloat/multilingual-e5-base",
    "twitter":"cardiffnlp/twitter-xlm-roberta-base",
}

RUN_DIR=Path("/content/v33_output")
RUN_DIR.mkdir(exist_ok=True)

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False

seed_all(SEED)
assert torch.cuda.is_available(),"Enable T4 GPU"

train=pd.read_csv(DATA_DIR/"train.csv",dtype={"ID":str})
valid=pd.read_csv(DATA_DIR/"validation.csv",dtype={"ID":str})
test=pd.read_csv(DATA_DIR/"test.csv",dtype={"ID":str})
champ=pd.read_csv(DATA_DIR/"submission_v14.csv",dtype={"ID":str})
v2v=pd.read_csv(DATA_DIR/"v2_valid.csv",dtype={"ID":str})
v2t=pd.read_csv(DATA_DIR/"v2_test.csv",dtype={"ID":str})

assert len(train)==3297 and len(valid)==500 and len(test)==1000
assert test.ID.tolist()==champ.ID.tolist()
assert test.ID.head(4).tolist()==[
    "BICH_0003","BICH_0006","BICH_0015","BICH_0016"
]

def clean(x):
    x=unicodedata.normalize("NFKC",str(x))
    x=x.replace("\u200c"," ").replace("\u200d"," ")
    return re.sub(r"\s+"," ",x).strip()

def twitter_clean(x):
    out=[]
    for t in clean(x).split():
        if t.startswith("@") and len(t)>1:
            out.append("@user")
        elif t.startswith("http"):
            out.append("http")
        else:
            out.append(t)
    return " ".join(out)

def key(x):
    return clean(x).lower()

for df in [train,valid,test]:
    df["raw_text"]=df.text.map(clean)
    df["twitter_text"]=df.text.map(twitter_clean)
    df["me5_text"]="query: "+df.raw_text
    df["group_key"]=df.text.map(key)

combined=pd.concat([train,valid],ignore_index=True)

print("✅ LIVE IDs verified")
display(pd.DataFrame({
    "train":train[LABELS].mean(),
    "validation":valid[LABELS].mean()
}).round(5))


In [ ]:

# 4. Recreate grouped 5-fold split on LIVE combined labels
groups=combined.groupby("group_key",sort=False).agg(
    size=("ID","size"),
    **{f"sum_{l}":(l,"sum") for l in LABELS}
).reset_index()

gp=np.array([combined[l].sum() for l in LABELS],float)
total=float(len(combined))
rarity=1/np.sqrt(np.maximum(gp,1))
mat=groups[[f"sum_{l}" for l in LABELS]].to_numpy(float)

rng=np.random.default_rng(SEED)
priority=(mat*rarity).sum(1)+.005*groups["size"].to_numpy()
order=np.argsort(-(priority+rng.random(len(groups))*1e-8))

fp=np.zeros((5,7))
fs=np.zeros(5)
assign={}

for idx in order:
    row=groups.iloc[idx]
    vec=np.array([row[f"sum_{l}"] for l in LABELS],float)
    size=float(row["size"])
    costs=[]

    for f in range(5):
        p=fp.copy(); s=fs.copy()
        p[f]+=vec; s[f]+=size

        costs.append(
            np.mean((p/np.maximum(gp[None,:],1)-.2)**2)
            +.30*np.mean((s/total-.2)**2)
        )

    best=int(np.argmin(costs))
    assign[row.group_key]=best
    fp[best]+=vec
    fs[best]+=size

combined["fold"]=combined.group_key.map(assign).astype(int)

assert combined.groupby("group_key").fold.nunique().max()==1
print(combined.groupby("fold").size().to_dict())


In [ ]:

# 5. Dataset + hierarchical representation head
class DS(Dataset):
    def __init__(self,df,tok,text_col,labels=True):
        self.x=df[text_col].astype(str).tolist()
        self.y=df[LABELS].to_numpy(np.float32) if labels else None
        self.tok=tok

    def __len__(self):
        return len(self.x)

    def __getitem__(self,i):
        e=self.tok(
            self.x[i],truncation=True,max_length=MAX_LENGTH,padding=False
        )
        if self.y is not None:
            e["labels"]=self.y[i]
        return e

class Coll:
    def __init__(self,tok):
        self.tok=tok

    def __call__(self,items):
        y=None
        if "labels" in items[0]:
            y=torch.tensor(
                np.stack([x.pop("labels") for x in items]),
                dtype=torch.float32
            )

        b=self.tok.pad(
            items,padding=True,pad_to_multiple_of=8,return_tensors="pt"
        )
        if y is not None:
            b["labels"]=y
        return b

class HierHead(nn.Module):
    def __init__(self,name):
        super().__init__()
        self.base=AutoModel.from_pretrained(name)
        h=self.base.config.hidden_size

        self.att=nn.Linear(h,1)
        self.toxic=nn.Linear(h*2,1)

        self.attrs=nn.Sequential(
            nn.Linear(h*2+1,h),
            nn.GELU(),
            nn.Dropout(.15),
            nn.Linear(h,6)
        )

    def forward(self,input_ids,attention_mask,token_type_ids=None):
        kw={"input_ids":input_ids,"attention_mask":attention_mask}
        if token_type_ids is not None:
            kw["token_type_ids"]=token_type_ids

        out=self.base(**kw)
        hs=out.last_hidden_state

        cls=hs[:,0]

        scores=self.att(hs).squeeze(-1)
        scores=scores.masked_fill(attention_mask==0,-1e4)
        a=torch.softmax(scores,dim=1).unsqueeze(-1)
        pooled=(hs*a).sum(1)

        rep=torch.cat([cls,pooled],dim=1)
        tox=self.toxic(rep)
        attrs=self.attrs(torch.cat([rep,tox.detach()],dim=1))

        return torch.cat([tox,attrs],dim=1)

def loss_fn(logits,y):
    tox=torch.nn.functional.binary_cross_entropy_with_logits(
        logits[:,0],y[:,0],reduction="none"
    )

    raw=torch.nn.functional.binary_cross_entropy_with_logits(
        logits[:,1:],y[:,1:],reduction="none"
    )

    # On this LIVE split every attribute positive implies toxic=1.
    # Keep toxic negatives fully; downweight easy non-toxic negatives.
    toxic=y[:,0:1]
    easy=torch.tensor(
        [.30,.35,.10,.10,.12,.12],
        dtype=torch.float32,
        device=logits.device
    ).unsqueeze(0)

    negw=toxic+(1-toxic)*easy
    w=torch.where(y[:,1:]>.5,torch.ones_like(raw),negw)

    posboost=torch.tensor(
        [1.35,1.20,1.10,1.40,1.15,1.35],
        dtype=torch.float32,
        device=logits.device
    ).unsqueeze(0)
    w=torch.where(y[:,1:]>.5,w*posboost,w)

    attrs=(raw*w).sum(1)/w.sum(1).clamp_min(1e-6)

    p=torch.sigmoid(logits.float())
    hierarchy=torch.relu(p[:,1:]-p[:,:1]).pow(2).mean(1)

    return (.50*tox+.50*attrs+.03*hierarchy).mean()

class FGM:
    def __init__(self,model,eps=FGM_EPS):
        self.model=model
        self.eps=eps
        self.backup={}

    def attack(self):
        for n,p in self.model.named_parameters():
            if (
                p.requires_grad and p.grad is not None
                and "word_embeddings" in n
            ):
                norm=torch.norm(p.grad)
                if torch.isfinite(norm) and norm>0:
                    self.backup[n]=p.data.clone()
                    p.data.add_(self.eps*p.grad/norm)

    def restore(self):
        for n,p in self.model.named_parameters():
            if n in self.backup:
                p.data=self.backup[n]
        self.backup={}

def llrd_groups(model):
    n_layers=int(getattr(model.base.config,"num_hidden_layers",12))
    groups={}

    for name,p in model.named_parameters():
        if not p.requires_grad:
            continue

        if not name.startswith("base."):
            lr=HEAD_LR
        else:
            m=re.search(r"encoder\.layer\.(\d+)",name)
            if m:
                layer=int(m.group(1))
                lr=BASE_LR*(LAYER_DECAY**(n_layers-1-layer))
            else:
                lr=BASE_LR*(LAYER_DECAY**n_layers)

        no_decay=(
            name.endswith("bias")
            or "LayerNorm.weight" in name
            or "layer_norm.weight" in name
        )
        wd=0.0 if no_decay else WEIGHT_DECAY
        groups.setdefault((lr,wd),[]).append(p)

    return [
        {"params":ps,"lr":lr,"weight_decay":wd}
        for (lr,wd),ps in groups.items()
    ]

@torch.no_grad()
def predict(model,loader):
    model.eval()
    parts=[]

    for b in loader:
        b.pop("labels",None)
        inp={k:v.to("cuda",non_blocking=True) for k,v in b.items()}

        with torch.autocast(device_type="cuda",dtype=torch.float16):
            logits=model(**inp)

        parts.append(torch.sigmoid(logits.float()).cpu().numpy())

    return np.concatenate(parts)


In [ ]:

# 6. Train one new expert — combined LIVE 5-fold OOF
fold_seeds=[2026,17,3407,1337,42]

def train_expert(expert):
    name=CHECKPOINTS[expert]
    text_col="me5_text" if expert=="me5" else "twitter_text"

    oof=np.zeros((len(combined),7),np.float32)
    test_parts=[]
    history=[]

    tok=AutoTokenizer.from_pretrained(name)

    for fold in range(5):
        tr=combined[combined.fold!=fold].copy()
        va=combined[combined.fold==fold].copy()

        cache=RUN_DIR/f"{expert}_fold_{fold}.npz"
        hp=RUN_DIR/f"{expert}_fold_{fold}_history.csv"

        if cache.exists():
            with np.load(cache) as z:
                idx=z["idx"]; pv=z["valid"]; pt=z["test"]

            if np.array_equal(idx,va.index.to_numpy()):
                print("✅ reuse",expert,fold)
                oof[idx]=pv
                test_parts.append(pt)
                continue

        print("\n"+"="*72)
        print(expert,"fold",fold)

        seed_all(fold_seeds[fold])
        gc.collect()
        torch.cuda.empty_cache()

        col=Coll(tok)

        tl=DataLoader(
            DS(tr,tok,text_col,True),
            batch_size=BATCH,shuffle=True,num_workers=0,pin_memory=True,
            collate_fn=col
        )
        vl=DataLoader(
            DS(va,tok,text_col,True),
            batch_size=EVAL_BATCH,shuffle=False,num_workers=0,pin_memory=True,
            collate_fn=col
        )
        tel=DataLoader(
            DS(test,tok,text_col,False),
            batch_size=EVAL_BATCH,shuffle=False,num_workers=0,pin_memory=True,
            collate_fn=col
        )

        model=HierHead(name).to("cuda")

        opt=torch.optim.AdamW(llrd_groups(model))
        steps=math.ceil(len(tl)/ACCUM)*EPOCHS
        sched=get_linear_schedule_with_warmup(
            opt,int(steps*WARMUP_RATIO),steps
        )
        scaler=torch.amp.GradScaler("cuda")
        fgm=FGM(model)

        sv=[]; st=[]; fh=[]

        for epoch in range(1,EPOCHS+1):
            model.train()
            opt.zero_grad(set_to_none=True)

            run=0.; seen=0

            for step,b in enumerate(
                tqdm(tl,desc=f"{expert} f{fold} e{epoch}/{EPOCHS}")
            ):
                y=b.pop("labels").to("cuda",non_blocking=True)
                inp={k:v.to("cuda",non_blocking=True) for k,v in b.items()}

                with torch.autocast(device_type="cuda",dtype=torch.float16):
                    logits=model(**inp)
                    loss=loss_fn(logits,y)

                scaler.scale(loss/ACCUM).backward()

                # Light FGM on epochs 2-3. No unscale here.
                if epoch>=2 and step%3==0:
                    fgm.attack()

                    with torch.autocast(device_type="cuda",dtype=torch.float16):
                        adv=loss_fn(model(**inp),y)

                    fgm.restore()
                    scaler.scale((.25*adv)/ACCUM).backward()

                run+=float(loss.item())*len(y)
                seen+=len(y)

                if (step+1)%ACCUM==0 or step+1==len(tl):
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(),1.)

                    old=scaler.get_scale()
                    scaler.step(opt)
                    scaler.update()

                    if scaler.get_scale()>=old:
                        sched.step()

                    opt.zero_grad(set_to_none=True)

            rec={
                "expert":expert,"fold":fold,
                "epoch":epoch,"loss":run/max(seen,1)
            }
            print(rec)
            fh.append(rec); history.append(rec)
            pd.DataFrame(fh).to_csv(hp,index=False)

            if epoch in SNAPSHOT_EPOCHS:
                sv.append(predict(model,vl))
                st.append(predict(model,tel))

        pv=np.mean(sv,axis=0)
        pt=np.mean(st,axis=0)

        oof[va.index.to_numpy()]=pv
        test_parts.append(pt)

        np.savez_compressed(
            cache,idx=va.index.to_numpy(),valid=pv,test=pt
        )

        del model,opt,sched,scaler,fgm,tl,vl,tel
        gc.collect()
        torch.cuda.empty_cache()

    ptest=np.mean(test_parts,axis=0)

    np.savez_compressed(
        RUN_DIR/f"{expert}_oof_test.npz",oof=oof,test=ptest
    )

    pd.DataFrame(history).to_csv(
        RUN_DIR/f"{expert}_training_history.csv",index=False
    )

    return oof,ptest


In [ ]:

# 7. Train mE5 and Twitter-XLM-R
me5_oof,me5_test=train_expert("me5")
tw_oof,tw_test=train_expert("twitter")

print("✅ New V33 experts complete")


In [ ]:

# 8. Load V32 + older LIVE reusable signals
v32=np.load(DATA_DIR/"v32_new_probabilities.npz")
v15=np.load(DATA_DIR/"v15_specialist.npz")
v17b=np.load(DATA_DIR/"v17_bangla_toxic.npz")
v17x=np.load(DATA_DIR/"v17_xlmr_toxic.npz")
v18=np.load(DATA_DIR/"v18_muril.npz")
v19=np.load(DATA_DIR/"v19_mdeberta_toxic.npz")
v23=np.load(DATA_DIR/"v23_conditional.npz")
v25=np.load(DATA_DIR/"v25_banglabert_large_toxic.npz")

oldb=np.load(DATA_DIR/"old_bangla_snapshots.npz")
oldx=np.load(DATA_DIR/"old_xlmr_snapshots.npz")

assert v32["stage2_oof"].shape==(3797,7)
assert me5_oof.shape==(3797,7)
assert tw_oof.shape==(3797,7)

print("✅ all LIVE signals aligned")


In [ ]:

# 9. Reconstruct V14 validation/test probabilities
def logit(p):
    p=np.clip(np.asarray(p,float),1e-6,1-1e-6)
    return np.log(p/(1-p))

def sigmoid(z):
    return 1/(1+np.exp(-z))

v14v={}
v14t={}

v14v["is_toxic"]=(
    .875*v2v.is_toxic_prob.to_numpy(float)
    +.125*oldx["valid"][1:3,:,0].mean(0)
)
v14t["is_toxic"]=(
    .875*v2t.is_toxic_prob.to_numpy(float)
    +.125*oldx["test"][1:3,:,0].mean(0)
)

j=LI["implicit_toxicity"]
v14v["implicit_toxicity"]=sigmoid(
    .9*logit(v2v.implicit_toxicity_prob)
    +.1*logit(oldb["valid"].mean(0)[:,j])
)
v14t["implicit_toxicity"]=sigmoid(
    .9*logit(v2t.implicit_toxicity_prob)
    +.1*logit(oldb["test"].mean(0)[:,j])
)

v14v["sarcasm"]=v2v.sarcasm_prob.to_numpy(float)
v14t["sarcasm"]=v2t.sarcasm_prob.to_numpy(float)

j=LI["contains_profanity"]
v14v["contains_profanity"]=(
    .95*v2v.contains_profanity_prob.to_numpy(float)
    +.05*oldb["valid"][0,:,j]
)
v14t["contains_profanity"]=(
    .95*v2t.contains_profanity_prob.to_numpy(float)
    +.05*oldb["test"][0,:,j]
)

j=LI["threatening"]
v14v["threatening"]=(
    .95*v2v.threatening_prob.to_numpy(float)
    +.05*oldx["valid"][0,:,j]
)
v14t["threatening"]=(
    .95*v2t.threatening_prob.to_numpy(float)
    +.05*oldx["test"][0,:,j]
)

j=LI["identity_attack"]
v14v["identity_attack"]=(
    .70*v2v.identity_attack_prob.to_numpy(float)
    +.30*oldb["valid"][2,:,j]
)
v14t["identity_attack"]=(
    .70*v2t.identity_attack_prob.to_numpy(float)
    +.30*oldb["test"][2,:,j]
)

v14v["sexual_content"]=v2v.sexual_content_prob.to_numpy(float)
v14t["sexual_content"]=v2t.sexual_content_prob.to_numpy(float)


In [ ]:

# 10. Train-only regularized super-stacker
NTR=len(train)
VIDX=np.arange(NTR,NTR+len(valid))

V15_LABELS=[
    "implicit_toxicity","contains_profanity","threatening","sexual_content"
]
V15I={l:i for i,l in enumerate(V15_LABELS)}
V23I={l:i for i,l in enumerate(ATTRS)}

def make_features(label,split):
    j=LI[label]

    if split=="train":
        sl=slice(0,NTR)
        new=[
            me5_oof[sl,j],tw_oof[sl,j],
            v32["stage2_oof"][sl,j],
            v32["lexical_oof"][sl,j],
        ]
    elif split=="validation":
        sl=VIDX
        new=[
            me5_oof[sl,j],tw_oof[sl,j],
            v32["stage2_oof"][sl,j],
            v32["lexical_oof"][sl,j],
        ]
    else:
        sl=None
        new=[
            me5_test[:,j],tw_test[:,j],
            v32["stage2_test"][:,j],
            v32["lexical_test"][:,j],
        ]

    if label=="is_toxic":
        if split=="test":
            new += [
                v32["stage1_test"],
                v17b["test"],v17x["test"],v19["test"],v25["test"],
                v18["test"][:,0],
            ]
        else:
            new += [
                v32["stage1_oof"][sl],
                v17b["oof"][sl],v17x["oof"][sl],v19["oof"][sl],v25["oof"][sl],
                v18["oof"][sl,0],
            ]
    else:
        if split=="test":
            new += [
                v18["test"][:,j],
                v23["test"][:,V23I[label]],
            ]
            if label in V15I:
                new += [v15["test"][:,V15I[label]]]
        else:
            new += [
                v18["oof"][sl,j],
                v23["oof"][sl,V23I[label]],
            ]
            if label in V15I:
                new += [v15["oof"][sl,V15I[label]]]

    raw=np.column_stack(new)
    return np.column_stack([raw,logit(raw)])

ytr=train[LABELS].to_numpy(int)
yva=valid[LABELS].to_numpy(int)

meta_val=np.zeros((len(valid),7),float)
meta_test=np.zeros((len(test),7),float)

for j,label in enumerate(LABELS):
    X=make_features(label,"train")
    Xv=make_features(label,"validation")
    Xt=make_features(label,"test")

    pipe=Pipeline([
        ("scale",StandardScaler()),
        ("lr",LogisticRegression(
            C=.05,solver="liblinear",max_iter=1000,random_state=SEED
        ))
    ])
    pipe.fit(X,ytr[:,j])

    meta_val[:,j]=pipe.predict_proba(Xv)[:,1]
    meta_test[:,j]=pipe.predict_proba(Xt)[:,1]

    print(label,"features",X.shape[1])

print("✅ V33 train-only stacker built")


In [ ]:

# 11. Exact competition metric helpers
def f1_pos(y,p):
    y=np.asarray(y,int); p=np.asarray(p,int)
    tp=((y==1)&(p==1)).sum()
    fp=((y==0)&(p==1)).sum()
    fn=((y==1)&(p==0)).sum()
    d=2*tp+fp+fn
    return float(2*tp/d) if d else 0.

def f1_macro(y,p):
    return float(np.mean([
        f1_pos((y==c).astype(int),(p==c).astype(int))
        for c in [0,1]
    ]))

def metric(label,y,p):
    return f1_macro(y,p) if label=="is_toxic" else f1_pos(y,p)

def official(y,p):
    ss=[metric(l,y[:,j],p[:,j]) for j,l in enumerate(LABELS)]
    return float(.5*ss[0]+.5*np.mean(ss[1:])),ss

def topk(prob,k):
    k=int(max(0,min(k,len(prob))))
    out=np.zeros(len(prob),int)
    if k:
        out[np.argsort(-prob)[:k]]=1
    return out

def best_topk(label,y,p,lo=None,hi=None):
    N=len(p)
    lo=0 if lo is None else max(0,int(lo))
    hi=N if hi is None else min(N,int(hi))

    order=np.argsort(-p)
    pred=np.zeros(N,int)

    # prefill to lo
    if lo>0:
        pred[order[:lo]]=1

    best=(metric(label,y,pred),lo)

    for k in range(lo+1,hi+1):
        pred[order[k-1]]=1
        s=metric(label,y,pred)
        if s>best[0]:
            best=(s,k)

    return best


In [ ]:

# 12. Conservative validation selection
# V14 remains available as fallback.
# Rank search is constrained around the V14 predicted count to avoid huge hidden-test shifts.

v14_thresholds={
    "is_toxic":0.6850110311061144,
    "implicit_toxicity":0.1416,
    "sarcasm":0.175,
    "contains_profanity":0.245,
    "threatening":0.16558668018085876,
    "identity_attack":0.3814,
    "sexual_content":0.245,
}

v14_val_pred=np.column_stack([
    (v14v[l]>=v14_thresholds[l]).astype(int)
    for l in LABELS
])

val_pred=v14_val_pred.copy()
test_pred=champ[LABELS].to_numpy(int).copy()

rows=[]
selected={}

for j,label in enumerate(LABELS):
    sources={
        "v14":(v14v[label],v14t[label]),
        "v33_meta":(meta_val[:,j],meta_test[:,j]),
        "me5":(me5_oof[VIDX,j],me5_test[:,j]),
        "twitter":(tw_oof[VIDX,j],tw_test[:,j]),
        "stage2":(v32["stage2_oof"][VIDX,j],v32["stage2_test"][:,j]),
        "lexical":(v32["lexical_oof"][VIDX,j],v32["lexical_test"][:,j]),
    }

    baseline_pred=v14_val_pred[:,j]
    baseline_score=metric(label,yva[:,j],baseline_pred)

    # Search counts only within ±20% of V14 validation positive count,
    # with a minimum width for rare labels.
    base_k=int(baseline_pred.sum())
    delta=max(3,int(round(.20*max(base_k,1))))
    lo=max(0,base_k-delta)
    hi=min(len(valid),base_k+delta)

    best={
        "key":(baseline_score,0,0),
        "name":"v14",
        "vp":baseline_pred.copy(),
        "tp":champ[label].to_numpy(int).copy(),
        "score":baseline_score,
        "k":base_k,
    }

    for name,(pv,pt) in sources.items():
        sc,k=best_topk(label,yva[:,j],pv,lo,hi)
        rate=k/len(valid)

        vp=topk(pv,k)
        test_k=int(round(rate*len(test)))
        tp=topk(pt,test_k)

        # require useful validation gain before replacing V14
        gain=sc-baseline_score

        rows.append({
            "label":label,"source":name,
            "validation_score":sc,
            "baseline_v14":baseline_score,
            "gain":gain,
            "validation_positive_count":k,
            "test_positive_count":test_k,
        })

        # tie-break to smaller change from V14
        changed=int((tp!=champ[label].to_numpy(int)).sum())
        key=(sc,-changed,-abs(k-base_k))

        if gain>=.006 and key>best["key"]:
            best={
                "key":key,"name":name,
                "vp":vp,"tp":tp,
                "score":sc,"k":k,
                "changed":changed,
            }

    selected[label]=best
    val_pred[:,j]=best["vp"]
    test_pred[:,j]=best["tp"]

selection=pd.DataFrame(rows)
selection.to_csv(
    RUN_DIR/"v33_validation_search.csv",index=False
)

score,scores=official(yva,val_pred)

report=pd.DataFrame({
    "label":LABELS,
    "selected_source":[selected[l]["name"] for l in LABELS],
    "validation_score":scores,
    "v14_score":[metric(l,yva[:,j],v14_val_pred[:,j]) for j,l in enumerate(LABELS)],
    "validation_pred_rate":val_pred.mean(0),
    "test_pred_rate":test_pred.mean(0),
    "test_changes_vs_v14":[
        int((test_pred[:,j]!=champ[l].to_numpy(int)).sum())
        for j,l in enumerate(LABELS)
    ],
})
report.to_csv(
    RUN_DIR/"v33_final_validation_report.csv",index=False
)

print("V33 validation official:",score)
display(report.round(5))


In [ ]:

# 13. Build LIVE candidate + strict ID checks
sub=champ.copy()

for j,l in enumerate(LABELS):
    sub[l]=test_pred[:,j].astype(int)

assert sub.ID.tolist()==test.ID.tolist()
assert sub.ID.tolist()==champ.ID.tolist()
assert sub.ID.head(4).tolist()==[
    "BICH_0003","BICH_0006","BICH_0015","BICH_0016"
]
assert sub[LABELS].isin([0,1]).all().all()

summary={
    "version":"v33_LIVE_me5_twitter_superensemble",
    "protected_public_lb":0.586,
    "validation_official_score":float(score),
    "validation_toxic_macro_f1":float(scores[0]),
    "validation_attribute_mean":float(np.mean(scores[1:])),
    "SUBMIT_CANDIDATE":bool(score>=.645),
    "note":"Higher gate than V32 because +0.002 validation did not transfer."
}

(RUN_DIR/"v33_summary.json").write_text(
    json.dumps(summary,indent=2)
)

sub.to_csv(
    RUN_DIR/"submission_v33_LIVE.csv",index=False
)

print(json.dumps(summary,indent=2))

if summary["SUBMIT_CANDIDATE"]:
    print("\n✅ V33 passed the stricter LIVE gate.")
else:
    print("\n❌ V33 did not pass the stricter gate. Send results ZIP; do not submit.")


In [ ]:

# 14. Package results
np.savez_compressed(
    RUN_DIR/"v33_new_probabilities.npz",
    me5_oof=me5_oof,me5_test=me5_test,
    twitter_oof=tw_oof,twitter_test=tw_test,
    meta_validation=meta_val,meta_test=meta_test
)

zip_path=Path("/content/v33_results.zip")

with zipfile.ZipFile(
    zip_path,"w",compression=zipfile.ZIP_DEFLATED
) as z:
    for p in RUN_DIR.iterdir():
        if p.is_file():
            z.write(p,arcname=p.name)

print("Created:",zip_path)

from google.colab import files
files.download(str(zip_path))
